# Titanic: Missing Values, Encoding, Scaling, and Similarity Measures

This notebook performs:
1. Loading the dataset and handling missing values (median for numeric, mode for categorical).
2. Categorical encodings: Label Encoding and One‑Hot Encoding.
3. Feature scaling: Min‑Max normalization and Z‑score standardization (numeric features only).
4. Similarity/Dissimilarity on rows: Pearson correlation (features), Cosine similarity (rows), Jaccard similarity (rows; one‑hot categoricals), and Euclidean distance (rows).

**Input file**: `/mnt/data/Lab2_titanic.csv`

Outputs (CSV files) are written next to this notebook in `/mnt/data/`.


In [3]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import re

INPUT_PATH = Path('Lab2_titanic.csv')  # change if needed
OUT_DIR = Path('/mnt/data')

df = pd.read_csv(INPUT_PATH)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1) Handle Missing Values
- **Numeric**: median
- **Categorical**: mode (or 'Unknown' if all values are missing)

In [10]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Impute numeric with median
df_num_imputed = df[numeric_cols].copy()
for c in numeric_cols:
    med = df_num_imputed[c].median()
    df_num_imputed[c] = df_num_imputed[c].fillna(med)

# Impute categorical with mode (or 'Unknown' if all NaN)
df_cat_imputed = df[categorical_cols].copy()
for c in categorical_cols:
    if df_cat_imputed[c].dropna().empty:
        df_cat_imputed[c] = df_cat_imputed[c].fillna('Unknown')
    else:
        mode_val = df_cat_imputed[c].mode(dropna=True)[0]
        df_cat_imputed[c] = df_cat_imputed[c].fillna(mode_val)

# Combine back, preserving original column order
df_clean = pd.concat([df_num_imputed, df_cat_imputed], axis=1)[df.columns]
print(df_clean.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare    Cabin Embarked  
0      0         A/5 21171   7.2500  B96 B98        S  
1      0          PC 17599  71.2833      C85        C  
2      0  STON/O2. 3101282   7.9250  B96 B98        S  
3      0            113803  53.1000     C123        S  
4      0            373450   8.0500  B96

## 2) Categorical Encoding
We will do both Label Encoding and One‑Hot Encoding.
- Label encoding creates an integer code per category (per column).
- One‑hot encoding expands each category to a 0/1 binary indicator.

In [12]:
# Label Encoding (adds <col>_label)
df_label = df_clean.copy()
for c in categorical_cols:
    df_label[c + '_label'] = pd.Categorical(df_label[c]).codes

df_label.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_label,Sex_label,Ticket_label,Cabin_label,Embarked_label
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,B96 B98,S,108,1,523,47,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,190,0,596,81,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,B96 B98,S,353,0,669,47,2
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,272,0,49,55,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,B96 B98,S,15,1,472,47,2


In [13]:
# One‑Hot Encoding
df_onehot = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=False, dtype=int)
df_onehot.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,"Name_Abbing, Mr. Anthony","Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,22.0,1,0,7.2500,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2,1,1,38.0,1,0,71.2833,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,1,3,26.0,0,0,7.9250,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,1,1,35.0,1,0,53.1000,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,5,0,3,35.0,0,0,8.0500,0,0,0,...,0,0,0,0,0,0,0,0,0,1


## 3) Feature Scaling (Numeric Only)
We compute:
- **Min‑Max Normalization**: $(x - \min)/(\max - \min)$
- **Z‑score Standardization**: $(x - \mu)/\sigma$

In [ ]:
X_num = df_clean[numeric_cols].astype(float).copy()

# Min‑Max
X_min = X_num.min()
X_max = X_num.max()
X_range = (X_max - X_min).replace(0, 1.0)
X_minmax = (X_num - X_min) / X_range
X_minmax_path = OUT_DIR / 'titanic_minmax_scaled.csv'
X_minmax.to_csv(X_minmax_path, index=False)
X_minmax.head()

In [ ]:
# Z‑score
X_mean = X_num.mean()
X_std = X_num.std(ddof=0).replace(0, 1.0)
X_zscore = (X_num - X_mean) / X_std
X_zscore_path = OUT_DIR / 'titanic_zscore_scaled.csv'
X_zscore.to_csv(X_zscore_path, index=False)
X_zscore.head()

## 4) Similarity & Dissimilarity Measures
We compute:
- **Pearson’s Correlation** (feature‑feature, numeric only)
- **Cosine Similarity** (row‑row, numeric standardized)
- **Jaccard Similarity** (row‑row, one‑hot categorical)
- **Euclidean Distance** (row‑row, numeric standardized)

For row‑wise matrices, we use the first 50 rows to keep the matrices a manageable size. Increase if you want.

In [ ]:
subset_size = min(50, len(df_clean))
index_labels = df_clean.index[:subset_size].astype(str)

# 4.1 Pearson correlation (feature‑feature)
pearson_corr = X_num.corr(method='pearson')
pearson_path = OUT_DIR / 'titanic_pearson_correlation_numeric_features.csv'
pearson_corr.to_csv(pearson_path)
pearson_corr.head()

In [ ]:
import numpy as np

# 4.2 Cosine similarity (row‑row, numeric standardized)
Xz_sub = X_zscore.iloc[:subset_size].to_numpy()
row_norms = np.linalg.norm(Xz_sub, axis=1, keepdims=True)
row_norms[row_norms == 0] = 1.0
Xz_unit = Xz_sub / row_norms
cosine_sim_matrix = Xz_unit @ Xz_unit.T

cosine_df = pd.DataFrame(cosine_sim_matrix, index=index_labels, columns=index_labels)
cosine_path = OUT_DIR / 'titanic_cosine_similarity_rows.csv'
cosine_df.to_csv(cosine_path)
cosine_df.iloc[:10, :10].round(3)

In [ ]:
# 4.3 Jaccard similarity (row‑row, one‑hot categorical)
if len(categorical_cols) > 0:
    joined = '|'.join(map(re.escape, categorical_cols))
    pattern = rf'^({joined})_'
    Xcat_sub = df_onehot.filter(regex=pattern).iloc[:subset_size].to_numpy(dtype=int)
else:
    Xcat_sub = np.empty((subset_size, 0), dtype=int)

if Xcat_sub.shape[1] > 0:
    A = Xcat_sub.astype(bool)
    inter = (A[:, None, :] & A[None, :, :]).sum(axis=2)
    sums = A.sum(axis=1, keepdims=True)
    union = sums + sums.T - inter
    with np.errstate(divide='ignore', invalid='ignore'):
        jaccard = np.where(union == 0, 1.0, inter / union)
    jaccard_df = pd.DataFrame(jaccard, index=index_labels, columns=index_labels)
else:
    jaccard_df = pd.DataFrame(np.nan, index=index_labels, columns=index_labels)

jaccard_path = OUT_DIR / 'titanic_jaccard_similarity_rows.csv'
jaccard_df.to_csv(jaccard_path)
jaccard_df.iloc[:10, :10].round(3)

In [ ]:
# 4.4 Euclidean distance (row‑row, numeric standardized)
diff = Xz_sub[:, None, :] - Xz_sub[None, :, :]
euclid = np.sqrt((diff ** 2).sum(axis=2))
euclid_df = pd.DataFrame(euclid, index=index_labels, columns=index_labels)
euclid_path = OUT_DIR / 'titanic_euclidean_distance_rows.csv'
euclid_df.to_csv(euclid_path)
euclid_df.iloc[:10, :10].round(3)

### Output Files
- `titanic_cleaned.csv`
- `titanic_label_encoded.csv`
- `titanic_onehot_encoded.csv`
- `titanic_minmax_scaled.csv`
- `titanic_zscore_scaled.csv`
- `titanic_pearson_correlation_numeric_features.csv`
- `titanic_cosine_similarity_rows.csv`
- `titanic_jaccard_similarity_rows.csv`
- `titanic_euclidean_distance_rows.csv`